# 03 - Your own measurements

A template for analysing **your own data** with the same workflow as notebooks
01 and 02.

It ships pointing at the nanopore data, so it runs out of the box and you can
see what each step does. To switch to your own measurement you change **one
block of file names** in section 2 - nothing else.

**Work through notebook 01 first.** The physics of each step is explained
there; here the focus is on plugging in new data safely.

In [ ]:
# Interactive plots. If they stay blank, use %matplotlib inline instead
# and restart the kernel.
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import hyperspy.api as hs
import exspy  # registers the EELS/EDX signal types

from workshop_data import (
    load_path,
    list_files,
    check_elements,
    check_background_window,
    energy_range,
)

print("HyperSpy", hs.__version__, "| exspy", exspy.__version__)

## 1. Put your files where the notebook can find them

Copy your measurement into the `data/` folder of this project, ideally in a
subfolder of its own:

```
data/
├── nanopore/            <- the workshop data
└── my_sample/           <- yours
    ├── EELS SI high-loss.dm4
    ├── EELS SI low-loss.dm4
    └── ADF.dm4
```

Anything HyperSpy can read works - `.dm3`, `.dm4`, `.emd`, `.hspy`, `.msa` and
more. The next cell lists what is actually there.

In [ ]:
# The underscore keeps Jupyter from echoing the returned list
# underneath the formatted output.
_ = list_files("*")

## 2. Point the notebook at your files

**This is the only block you need to edit.** Copy a path from the listing above
and paste it in. Set `LOW_LOSS` to `None` if you did not record a low-loss
spectrum - the alignment step is then skipped.

In [ ]:
# --- EDIT THIS BLOCK -------------------------------------------------------
HIGH_LOSS = "nanopore/EELS Spectrum Image (high-loss).dm4"
LOW_LOSS  = "nanopore/EELS Spectrum Image (low-loss).dm4"   # or None
SURVEY    = "nanopore/ADF Image.dm4"                        # or None

ELEMENTS  = ["Si", "O", "N"]        # what you expect in the sample
BINNING   = [2, 2, 1]              # [x, y, energy] - raise x/y to speed up fits
# ---------------------------------------------------------------------------

signal = load_path(HIGH_LOSS, signal_type="EELS")
ll = load_path(LOW_LOSS, signal_type="EELS") if LOW_LOSS else None

low, high = energy_range(signal)
print(f"{type(signal).__name__}  shape={signal.data.shape}")
print(f"energy axis: {low:.0f} - {high:.0f} eV")
print(f"low-loss: {'loaded' if ll is not None else 'not used'}")

In [ ]:
if SURVEY:
    load_path(SURVEY).plot()

## 3. Sanity checks before modelling

Two mistakes cost a lot of time because **neither of them raises an error**:

- `add_elements` silently drops every element whose edges lie outside the
  recorded energy window. The model then quietly omits them.
- `remove_background` accepts a window that is not inside the data at all and
  returns something meaningless.

The next two cells catch both. Run them whenever you change the data or the
element list.

In [ ]:
result = check_elements(signal, ELEMENTS)

# Keep only what is actually measurable
ELEMENTS = result["usable"]
print("\nusing:", ELEMENTS)

Now pick the background window. It has to sit **inside the measured range and
before the first edge you care about**. The cell prints the edges again so you
can choose a gap.

In [ ]:
# --- EDIT: background window in eV ---
BACKGROUND = (70.0, 96.0)
# -------------------------------------

print("edges in range:")
for element in ELEMENTS:
    for name, energy in result["edges"][element]:
        print(f"  {element:3s} {name:6s} {energy:7.0f} eV")
print()
check_background_window(signal, BACKGROUND)

## 4. Align the zero-loss peak

Only possible with a low-loss acquisition. Skipped automatically otherwise -
your edges may then sit at slightly wrong energies.

In [ ]:
if ll is not None:
    ll.align_zero_loss_peak(also_align=[signal], signal_range=(-10.0, 10.0))
    print("aligned")
else:
    print("no low-loss data - skipping alignment")

## 5. Build and fit the model

The first `create_model()` on a new machine downloads the GOSH database once
(~42 MB). `multifit` fits every pixel, so raise `BINNING` if it takes too long.

In [ ]:
signal.add_elements(ELEMENTS)

signal_binned = signal.rebin(scale=BINNING)
signal_binned = signal_binned.remove_background(signal_range=BACKGROUND)

m = signal_binned.create_model(auto_background=False)
m.components

In [ ]:
# --- Variant A: interactive ---
m.gui()

In [ ]:
# --- Variant B: same thing in code ---
for component in m:
    print(f"{component.name}   active={component.active}")

In [ ]:
m.plot()

In [ ]:
m.multifit(kind="smart")

In [ ]:
m.plot_results()

## 6. Optional: fine structure with your own reference spectra

If you measured reference spectra, put them in their own folder under `data/`
and set `STANDARDS_FOLDER`. Set it to `None` to skip this section.

Each reference becomes a `ScalableFixedPattern` whose height is fitted; the
fitted height is that reference's share in the pixel. Important: the references
must get **exactly the same preprocessing as the data**, otherwise you compare
background-carrying curves against background-free ones.

In [ ]:
# --- EDIT THIS BLOCK -------------------------------------------------------
STANDARDS_FOLDER = "nanopore/Si Standards"   # or None to skip
FIT_WINDOW       = (92.0, 170.0)             # region to fit, in eV
SMOOTHING        = 2                         # Gaussian sigma, in channels
# ---------------------------------------------------------------------------

if STANDARDS_FOLDER:
    from workshop_data import load_standards

    binned = signal.rebin(scale=BINNING)
    binned = binned.remove_background(signal_range=BACKGROUND)
    binned = binned.isig[FIT_WINDOW[0]:FIT_WINDOW[1]]

    s_smooth = binned.deepcopy()
    s_smooth.data = gaussian_filter1d(s_smooth.data, sigma=SMOOTHING, axis=-1)

    standards = load_standards(folder=STANDARDS_FOLDER, sigma=SMOOTHING)
    for name, s in list(standards.items()):
        s.data = s.data / s.data.max()
        print(f"{name:16s} {s.axes_manager[-1].size} channels")
else:
    print("no reference spectra - skipping section 6")

In [ ]:
if STANDARDS_FOLDER:
    # A judgement call. With auto_add_edges=True (the default) the model holds
    # both the physical edges and your references - that is what notebook 01 does
    # for the Si standards. If your references already describe the whole edge,
    # set auto_add_edges=False, otherwise the same feature is modelled twice and
    # the fit becomes ambiguous.
    m = s_smooth.create_model(auto_background=False, auto_add_edges=True)

    for name, s in standards.items():
        pattern = hs.model.components1D.ScalableFixedPattern(s)
        pattern.name = name
        pattern.xscale.free = False
        pattern.shift.free = False
        pattern.yscale.bmin = 0
        pattern.yscale.bmax = 1e7
        m.append(pattern)

    display(m.components)

In [ ]:
if STANDARDS_FOLDER:
    m.multifit(bounded=True)
    m.plot_results()

## 7. Give your dataset a permanent name

Once you use a file more than once, a literal path in the notebook gets in the
way. Add a line to `DATASETS` in `notebooks/workshop_data.py`:

```python
DATASETS = {
    ...
    "my_sample_eels": "my_sample/EELS SI high-loss.dm4",
}
```

After that, `load("my_sample_eels")` works in **every** notebook, and
`pixi run check` verifies the file is present before the session starts.

If the file is not part of the shared data ZIP, also add its name to
`OPTIONAL` in the same file - then a missing file is reported as a note rather
than an error.

## For EDX instead of EELS

Same idea, three changes:

```python
edx = load_path("my_sample/EDS SI.dm4", signal_type="EDS_TEM")
edx.set_elements(["Si", "N", "O"])
edx.add_lines()
```

`check_elements` does not apply - it looks at EELS ionisation edges. For EDX,
`add_lines()` picks the lines that fit the accelerating voltage, and
`print(edx.metadata.Sample.xray_lines)` shows what it chose. Notebook 02 has
the full workflow.